# AffectLab — IEMOCAP speaker-independent audio baseline

This notebook fine-tunes `facebook/wav2vec2-base` on the same five four-class IEMOCAP folds as the frozen text experiments. It downloads a private 595.6 MB task-specific audio bundle instead of the 17.7 GB raw release. Run fold 5 first. An A100 or L4 is recommended; T4 may require reducing batch size to 2 and increasing gradient accumulation to 8.

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess

from google.colab import auth

PROJECT_ID = 'cat-behaviour-research'
BUCKET = 'affectlab-research-raluca-biras'
TEXT_DATA_GCS = f'gs://{BUCKET}/data/processed/iemocap-text-v1'
AUDIO_GCS = f'gs://{BUCKET}/data/processed/iemocap-audio-benchmark4-v1'
RUNS_GCS = f'gs://{BUCKET}/runs/iemocap-audio'
auth.authenticate_user()
subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID], check=True)

In [ ]:
import base64
import hashlib
import json
import os
import sys
from pathlib import Path

from google.colab import userdata

REPO_URL = 'https://github.com/ralucabiras/emotion-aware-role-play-model.git'
REPO_DIR = Path('/content/emotion-aware-role-play-model')
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets and enable notebook access.')
basic_auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
auth_option = f'http.extraHeader=Authorization: Basic {basic_auth}'
if not REPO_DIR.exists():
    subprocess.run(['git', '-c', auth_option, 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), '-c', auth_option, 'pull', '--ff-only'], check=True)
del github_token, basic_auth, auth_option
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-ml.txt')], check=True)
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

In [ ]:
DATA_ROOT = Path('/content/iemocap-text-v1')
AUDIO_ROOT = Path('/content/iemocap-audio')
AUDIO_BUNDLE = Path('/content/iemocap-benchmark4-audio-v1.tar.gz')
AUDIO_MANIFEST = Path('/content/iemocap-benchmark4-audio-v1.tar.gz.manifest.json')
OUTPUT_ROOT = Path('/content/iemocap-audio-runs')
subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', TEXT_DATA_GCS, str(DATA_ROOT)], check=True)
subprocess.run(['gcloud', 'storage', 'cp', f'{AUDIO_GCS}/{AUDIO_MANIFEST.name}', str(AUDIO_MANIFEST)], check=True)
manifest = json.loads(AUDIO_MANIFEST.read_text())
if not AUDIO_BUNDLE.exists():
    subprocess.run(['gcloud', 'storage', 'cp', f'{AUDIO_GCS}/{AUDIO_BUNDLE.name}', str(AUDIO_BUNDLE)], check=True)
assert AUDIO_BUNDLE.stat().st_size == manifest['bundle_bytes']
digest = hashlib.sha256()
with AUDIO_BUNDLE.open('rb') as handle:
    while chunk := handle.read(16 * 1024 * 1024):
        digest.update(chunk)
assert digest.hexdigest().upper() == manifest['bundle_sha256']
assert manifest['sample_rate_counts'] == {'16000': 5531}
from ml.preprocessing.iemocap_audio import extract_bundle

if not (AUDIO_ROOT / 'audio').exists():
    extract_bundle(AUDIO_BUNDLE, AUDIO_ROOT, manifest['files'])
assert len(list((AUDIO_ROOT / 'audio').glob('*.wav'))) == 5531
print('Verified audio bundle:', manifest)

## Train

Run fold 5 first. Audio training is slower than text. After reviewing fold 5, set `RUN_ALL_AUDIO_FOLDS = True` and rerun this cell. Results upload after each completed fold.

In [ ]:
SEED = 42
RUN_ALL_AUDIO_FOLDS = False
folds = range(1, 6) if RUN_ALL_AUDIO_FOLDS else [5]
audio_config = REPO_DIR / 'configs' / 'iemocap_audio_wav2vec2_base.json'
audio_experiment = 'iemocap_benchmark4_audio_wav2vec2_base'
for fold in folds:
    print(f'\n=== Audio benchmark fold {fold} ===')
    subprocess.run(
        [sys.executable, '-m', 'ml.training.train_iemocap_audio', '--config', str(audio_config),
         '--data-root', str(DATA_ROOT), '--audio-root', str(AUDIO_ROOT), '--output-root', str(OUTPUT_ROOT),
         '--fold', str(fold), '--seed', str(SEED)],
        check=True,
    )
    local_fold = OUTPUT_ROOT / audio_experiment / f'fold-{fold}'
    subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', str(local_fold), f'{RUNS_GCS}/{audio_experiment}/fold-{fold}'], check=True)

In [ ]:
if RUN_ALL_AUDIO_FOLDS:
    experiment_dir = OUTPUT_ROOT / audio_experiment
    subprocess.run([sys.executable, '-m', 'ml.evaluation.summarize_iemocap_folds', str(experiment_dir)], check=True)
    summary = json.loads((experiment_dir / 'summary.json').read_text())
    display(summary)
    subprocess.run(['gcloud', 'storage', 'cp', str(experiment_dir / 'summary.json'), f'{RUNS_GCS}/{audio_experiment}/summary.json'], check=True)
else:
    metrics = json.loads((OUTPUT_ROOT / audio_experiment / 'fold-5' / 'metrics.json').read_text())
    display(metrics['audio_audit'])
    display(metrics['validation_metrics'])
    display(metrics['test_metrics'])
    display(metrics['calibration'])
    display(metrics['test_per_class'])

## Guardrails

- Keep the audio bundle, row-level predictions, and checkpoints private.
- Do not compare one audio fold with the pooled text result.
- Do not tune hyperparameters against held-out test sessions.
- Fusion begins only after all five audio folds and the pooled summary are complete.